# Temporary M5 region-mask branch validation

Test one existing Peru model/region/period/CIM cell using the feature branch, without overwriting production classification paths or assets. The initial pilot uses model `training_001_r1_v1_sentinel2`, region `peru_r1_costa_norte`, period `2025_01`, cell `SA-17-Z-C`. It writes a unique `_VALIDATION` GCS object and creates then force-overwrites a uniquely named temporary GEE image asset.

Open this notebook from the feature branch in Colab; no merge is required. This test uses all bands configured for the selected model. Do not set `RUN_TEST=True` until selectors identify a suitable tile and the temporary GEE asset path is acceptable.

In [ ]:
import os, subprocess, sys
REPO_DIR = '/content/brazil-fire-region-test'
BRANCH = 'fix/v04-region-only-classification'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/mapbiomas/brazil-fire.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'FETCH_HEAD'], check=True)
CORE = os.path.join(REPO_DIR, 'network/catalog/development/version_04/src/core')
sys.path.insert(0, CORE)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())
# Colab dependencies; install GDAL bindings as in the country notebooks.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'gdal-bin', 'python3-gdal'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'earthengine-api', 'google-cloud-storage', 'gcsfs', 'rasterio', 'numpy<2', 'shapely>=2', 'pyproj>=3.6', 'scipy', 'tqdm'], check=True)

## Configure the one-tile test
Set the selectors from an existing M5 model and COG period. Choose a cell intersecting the region with valid source pixels. The first run authenticates using the account already authorized for the Peru project/bucket.

In [ ]:
COUNTRY = 'peru'
CAMPAIGN = 'MONITOR_01'
MODEL_ID = 'training_001_r1_v1_sentinel2'
REGION_NAME = 'peru_r1_costa_norte'  # exact region_nam in the campaign asset
PERIOD = '2025_01'
CELL_ID = 'SA-17-Z-C'  # existing M5 output tile; test writes to _VALIDATION, never its production path
RUN_TEST = False    # explicit write gate; leave False until selectors are checked

from M0_auth_config import set_global_opts, authenticate, CONFIG
set_global_opts(
    country=COUNTRY, campaign=CAMPAIGN, sensor='sentinel2', periodicity='monthly',
    mosaic_methods=['minnbr', 'minnbr_buffer'], personal_task_flag='MONITOR', language='es',
    gcs_bucket='mapbiomas-fire',
    gcs_library_images_prefix=f'sudamerica/{COUNTRY}/CATALOG_01',
    gcs_campaigns_prefix=f'sudamerica/{COUNTRY}/CATALOG_01',
    gee_project=f'mapbiomas-{COUNTRY}',
    gee_library_images_prefix=f'projects/mapbiomas-{COUNTRY}/assets/FIRE/CATALOG_01',
    gee_campaigns_prefix=f'projects/mapbiomas-{COUNTRY}/assets/FIRE/CATALOG_01',
    asset_regions=f'projects/mapbiomas-{COUNTRY}/assets/FIRE/CATALOG_01/{CAMPAIGN}/AUXILIARY/regiones_fuego_{COUNTRY}_v1',
)
if RUN_TEST:
    assert all([MODEL_ID, REGION_NAME, PERIOD, CELL_ID]), 'Set all four selectors before enabling RUN_TEST.'
    authenticate()
    from M0_auth_config import _get_fs
    from M5_classifier import _get_region_cells, _get_region_geometry
    assert any(c['name'] == CELL_ID for c in _get_region_cells(REGION_NAME)), 'CELL_ID is not an intersecting region cell.'
    print('Preflight passed; GCS:', CONFIG['bucket'], 'region:', REGION_NAME, 'cell:', CELL_ID, 'period:', PERIOD)
else:
    print('Dry/preflight only. Set selectors, inspect the target, then explicitly set RUN_TEST=True.')

In [ ]:
if RUN_TEST:
    import uuid, tempfile, rasterio, numpy as np, ee
    from M5_inference import load_model_from_gcs, build_band_paths, classify_cell_with_cogs
    from M0_auth_config import _gcs_models_base, get_temp_dir, _gcs_download
    from M5_spatial import clip_geometry_to_cell, rasterize_region_mask
    from M6_image_upload import build_image_upload_command
    from M5_workplan import gcs_full, gcs_classifications_path
    from M5_classifier import _get_region_geometry
    fs = _get_fs()
    model_dir = f"gs://{CONFIG['bucket']}/{_gcs_models_base()}/{MODEL_ID}"
    predict_fn, meta, bands_config, norm_stats, band_order = load_model_from_gcs(model_dir, fs)
    band_paths = build_band_paths(bands_config, PERIOD)
    missing = [p for p in band_paths.values() if not fs.exists(f'gs://{p}')]
    assert not missing, f'Missing source COG(s): {missing}'
    region_geojson = _get_region_geometry(REGION_NAME)
    cim = ee.FeatureCollection('projects/mapbiomas-workspace/AUXILIAR/cim-world-1-250000')
    cell_geojson = cim.filter(ee.Filter.eq('name', CELL_ID)).first().geometry().getInfo()
    run_id = uuid.uuid4().hex
    staging_key = f"{gcs_classifications_path()}/_VALIDATION/M5_region_mask/{MODEL_ID}/{REGION_NAME}/{CELL_ID}_{PERIOD}_{run_id}.tif"
    staging_path = gcs_full(staging_key)
    result = classify_cell_with_cogs(
        CELL_ID, predict_fn, bands_config, norm_stats, staging_path, band_paths, band_order,
        region_geometry=region_geojson, logger=print,
    )
    assert result is not None, 'No valid pixels in this cell/region/period; choose another test cell.'
    local_path = os.path.join(get_temp_dir('tiles'), f'validation_{uuid.uuid4().hex}.tif')
    _gcs_download(staging_path, local_path)
    with rasterio.open(local_path) as src:
        prob = src.read(1)
        clipped, _ = clip_geometry_to_cell(region_geojson, cell_geojson, src.crs)
        inside = rasterize_region_mask(clipped, prob.shape, src.transform)
    outside_classified = int(((prob != -9999) & ~inside).sum())
    classified_pixels = int((prob != -9999).sum())
    assert outside_classified == 0, f'{outside_classified} pixels outside the region were classified.'
    assert classified_pixels == result['total_pixels'], 'Raster count differs from M5 valid-pixel statistics.'
    from M0_auth_config import _gcs_upload
    _gcs_upload(local_path, staging_path)
    ee_parent = f"{CONFIG['gee_campaigns_prefix']}/{CAMPAIGN}/LIBRARY_CLASSIFICATIONS/REGIONAL/{MODEL_ID}"
    ee.data.getAsset(ee_parent)  # Require the existing model collection; do not create production folders here.
    validation_asset = f"{ee_parent}/_VALIDATION_{CELL_ID}_{PERIOD}_{run_id}"
    time_meta = __import__('M6_publisher')._period_export_metadata(PERIOD)
    command = build_image_upload_command(f'mapbiomas-{COUNTRY}', validation_asset, f'gs://{staging_path}', {
        **time_meta, 'source': 'mapbiomas-fire-validation', 'model_id': MODEL_ID,
        'region': REGION_NAME, 'period': PERIOD, 'year': time_meta['year'],
        'month': time_meta['month'], 'campaign': CAMPAIGN, 'country': COUNTRY,
        'version': 'version_04', 'type': time_meta['asset_type'],
    })
    command.insert(4, '--wait')  # Wait for ingestion to finish for this controlled test.
    first = subprocess.run(command, check=True, capture_output=True, text=True)
    second = subprocess.run(command, check=True, capture_output=True, text=True)  # Exercise --force on the same validation ID.
    asset = ee.data.getAsset(validation_asset)
    assert asset['name'] == validation_asset, 'Validation asset was not readable after overwrite.'
    assert ee.Image(validation_asset).bandNames().getInfo() == ['probability', 'dayOfYear']
    print({'PASS': True, 'classified_pixels': classified_pixels, 'outside_region_pixels': outside_classified, 'gcs_validation_tif': staging_path, 'temporary_gee_asset': validation_asset, 'first_registration': first.stdout[-500:], 'overwrite_registration': second.stdout[-500:]})
    print('No production tile or GEE asset was changed. Remove the uniquely named validation GCS objects and temporary GEE asset manually after review.')